questo blocco di codice serve a importare tutti gli strumenti necessari per costruire e addestrare una rete neurale per la computer vision.

In [ ]:
import torch
# Importa la libreria principale PyTorch che fornisce funzionalità base per il deep learning come tensori, autograd, e operazioni su GPU

from torch import nn
# from torch: dal pacchetto principale di PyTorch
# import nn: importa il modulo "nn" (neural networks) che contiene classi per costruire reti neurali come layer, funzioni di attivazione e contenitori sequenziali

from torch.utils.data import DataLoader
# from torch.utils.data: dal sottopacchetto "utils.data" di PyTorch
# - utils: è il sottopacchetto che contiene strumenti di utilità generale
# - data: è il modulo specifico per la gestione dei dati
# import DataLoader: importa la classe DataLoader che serve per caricare i dati in batch durante l'addestramento

from torchvision import datasets
# from torchvision: dal pacchetto torchvision, dedicato alla computer vision
# import datasets: importa il modulo che contiene dataset di immagini predefiniti e pronti all'uso come MNIST, CIFAR10, ImageNet

from torchvision.transforms import ToTensor
# from torchvision.transforms: dal sottopacchetto "transforms" di torchvision
# - transforms: contiene funzioni per trasformare e preprocessare immagini
# import ToTensor: importa la trasformazione specifica che converte immagini in formato PIL o array numpy in tensori PyTorch



Questo codice scarica il dataset Fashion-MNIST (immagini di vestiti, scarpe, borse in bianco e nero 28x28 pixel) e lo divide in:

training_data: 60.000 immagini per addestrare il modello

test_data: 10.000 immagini per testare il modello

I dati vengono salvati nella cartella "data" e convertiti automaticamente in tensori PyTorch

In [ ]:
# Download training data from open datasets.

training_data = datasets.FashionMNIST(
    # datasets.FashionMNIST: dal modulo torchvision.datasets, carica il dataset Fashion-MNIST (immagini di abbigliamento)

    root="data",
    # root: specifica la cartella dove salvare/scaricare i dati
    # "data": crea una cartella chiamata "data" nella directory corrente

    train=True,
    # train=True: carica la parte di TRAINING del dataset (60.000 immagini)
    # train=False: avrebbe caricato la parte di TEST (10.000 immagini)

    download=True,
    # download=True: scarica il dataset da internet se non è già presente nella cartella "data"

    transform=ToTensor(),
    # transform=ToTensor(): applica la trasformazione ToTensor() a ogni immagine
    # Converte le immagini PIL in tensori PyTorch e normalizza i pixel da [0,255] a [0.0,1.0]
)


# Download test data from open datasets.

test_data = datasets.FashionMNIST(
    # datasets.FashionMNIST: stesso dataset, ma per il TEST

    root="data",
    # root: stessa cartella "data" (così training e test sono insieme)

    train=False,
    # train=False: carica la parte di TEST (10.000 immagini)

    download=True,
    # download=True: scarica anche questi (di solito vengono insieme, ma meglio metterlo)

    transform=ToTensor(),
    # transform=ToTensor(): stessa trasformazione applicata ai dati di test
)

100%|██████████| 26.4M/26.4M [00:01<00:00, 16.6MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 305kB/s]
100%|██████████| 4.42M/4.42M [00:00<00:00, 5.58MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 11.2MB/s]


Prepara i dati per l'addestramento:

- Crea due DataLoader: uno per il training e uno per il test
- Imposta batch_size=64, quindi ogni batch conterrà 64 immagini con le relative etichette
- Il ciclo for finale mostra la struttura del primo batch per verificare che tutto funzioni

In [ ]:
batch_size = 64
# batch_size = 64: imposta a 64 il numero di campioni elaborati insieme in un singolo batch

# Create data loaders.

train_dataloader = DataLoader(training_data, batch_size=batch_size)
# train_dataloader: crea un DataLoader per i dati di addestramento
# DataLoader(training_data, batch_size=batch_size): prende il dataset training_data e lo configura per restituire batch da 64 elementi

test_dataloader = DataLoader(test_data, batch_size=batch_size)
# test_dataloader: crea un DataLoader per i dati di test
# DataLoader(test_data, batch_size=batch_size): stesso batch_size per i dati di test

for X, y in test_dataloader:
    # for X, y in test_dataloader: itera sul dataloader di test (prende solo il primo batch grazie a break)
    # X: contiene le immagini del batch (64 immagini)
    # y: contiene le etichette corrispondenti alle 64 immagini

    print(f"Shape of X [N, C, H, W]: {X.shape}")
    # print: mostra la forma del tensore X
    # [N, C, H, W]: formato standard per immagini in PyTorch
    # N = numero di campioni nel batch (64)
    # C = canali (1 per Fashion-MNIST perché sono in bianco e nero)
    # H = altezza dell'immagine (28 pixel)
    # W = larghezza dell'immagine (28 pixel)

    print(f"Shape of y: {y.shape} {y.dtype}")
    # print: mostra la forma del tensore y e il tipo di dato
    # y.shape: sarà [64] (un'etichetta per ogni immagine)
    # y.dtype: intero (le etichette sono numeri interi da 0 a 9)

    break
    # break: esce dal ciclo dopo il primo batch (serve solo per vedere com'è fatto, non processa tutto)

Shape of X [N, C, H, W]: torch.Size([64, 1, 28, 28])
Shape of y: torch.Size([64]) torch.int64


**Creating Models**

In [ ]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
# device: determina se usare GPU o CPU per i calcoli
# torch.accelerator.current_accelerator().type: rileva il tipo di acceleratore disponibile (GPU, TPU, ecc.)
# if torch.accelerator.is_available(): verifica se c'è un acceleratore disponibile
# else "cpu": se non c'è acceleratore, usa la CPU
# Il risultato sarà "cuda" (GPU NVIDIA), "tpu" (TPU Google) o "cpu"

print(f"Using {device} device")
# print: mostra quale dispositivo viene utilizzato (es. "Using cuda device")



# Define model

class NeuralNetwork(nn.Module):
    # class NeuralNetwork: definisce una nuova classe chiamata NeuralNetwork
    # (nn.Module): eredita dalla classe Module di PyTorch (classe base per tutte le reti neurali)

    def __init__(self):
        # __init__: metodo costruttore, viene eseguito quando crei un'istanza della classe
        # self si riferirà all'oggetto che verrà creato, è il modo in cui un oggetto fa riferimento a se stesso.
        super().__init__()
        # super().__init__(): chiama il costruttore della classe padre (nn.Module)

        self.flatten = nn.Flatten()
        # self.flatten: attributo che contiene un layer Flatten
        # nn.Flatten(): layer che appiattisce immagini 2D (28x28) in un vettore 1D (784)

        self.linear_relu_stack = nn.Sequential(
            # self.linear_relu_stack: attributo che contiene una sequenza di layer
            # nn.Sequential(): contenitore che applica i layer in ordine
            nn.Linear(28*28, 512),
            # nn.Linear(28*28, 512): strato lineare che va da 784 neuroni a 512 neuroni
            nn.ReLU(),
            # nn.ReLU(): funzione di attivazione ReLU (rettificatore lineare)
            nn.Linear(512, 512),
            # nn.Linear(512, 512): secondo strato lineare da 512 a 512 neuroni
            nn.ReLU(),
            # nn.ReLU(): altra attivazione ReLU
            nn.Linear(512, 10)
            # nn.Linear(512, 10): strato finale da 512 a 10 neuroni (10 classi di output)
        )

        #in pratica self.linear_relu_stack prende la struttura di nn.Sequential in nn e la rinomina,
        #in sostanza cambiando delle parti tipo relu e linear dando un ordine a come la funzione diciamo self.linear_relu_stack
        #elaborerà i dati

    def forward(self, x):
        # forward: metodo definito da nn.Module, esegue il passaggio in avanti della rete
        # self: riferimento all'istanza corrente
        # x: input (le immagini in ingresso)

        x = self.flatten(x)
        # x = self.flatten(x): applica il layer flatten a x (appiattisce le immagini)

        logits = self.linear_relu_stack(x)
        # logits = self.linear_relu_stack(x): passa x attraverso la sequenza di layer lineari e ReLU
        # logits: output grezzi (non normalizzati) prima dell'applicazione di softmax

        return logits
        # return logits: restituisce i logits come output della rete

model = NeuralNetwork().to(device)
# model = NeuralNetwork(): crea un'istanza (oggetto) della classe NeuralNetwork
# .to(device): sposta il modello sul dispositivo specificato (GPU o CPU)
# model: variabile che contiene il modello pronto per l'addestramento

print(model)
# print(model): stampa la struttura del modello (mostra tutti i layer)